# 미들웨어 실습 — Before vs After로 체감하기

각 미들웨어를 **실무 활용도 순**으로 배치하고, 매번 **미들웨어 없이 vs 있을 때**를 비교합니다.

| 순서 | 미들웨어 | 활용도 | 한줄 설명 |
|------|----------|--------|----------|
| 1 | `ModelCallLimitMiddleware` | 필수 | LLM 호출 횟수 제한 (비용 폭주 방지) |
| 2 | `ToolCallLimitMiddleware` | 필수 | 도구 호출 횟수 제한 |
| 3 | `ModelFallbackMiddleware` | 높음 | 모델 장애 시 자동 전환 |
| 4 | `PIIMiddleware` | 높음 | 개인정보 자동 마스킹 |
| 5 | `SummarizationMiddleware` | 보통 | 긴 대화 자동 요약 (토큰 절약) |
| 6 | `HumanInTheLoopMiddleware` | 보통 | 위험 액션 실행 전 사람 승인 |

## 0. 환경 설정

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import time

load_dotenv(override=True)

model = ChatOpenAI(model="gpt-5.4")
print("환경 설정 완료")

환경 설정 완료


---
## 1. ModelCallLimitMiddleware (필수)

**LLM 호출 횟수를 제한**합니다. 에이전트가 무한루프에 빠지면 API 비용이 폭발하는데, 이걸 막아줍니다.

### 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `thread_limit` | 전체 대화(스레드)에서 최대 호출 수 | `thread_limit=50` |
| `run_limit` | 한 번의 run에서 최대 호출 수 | `run_limit=5` |
| `exit_behavior` | 제한 초과 시 행동 | `"end"` (정상 종료), `"error"` (예외 발생) |

### Before: 제한 없이 실행

검색 도구가 일부러 "결과 부족"만 반환 → LLM이 계속 재시도 → 호출 횟수 폭주

In [2]:
call_count_before = 0

@tool
def bad_search(query: str) -> str:
    """검색을 수행합니다. 반드시 이 도구로 검색해야 합니다."""
    global call_count_before
    call_count_before += 1
    print(f"  [도구 호출 {call_count_before}회]")
    return f"결과 부족 -- '{query}'에 대한 정보를 찾지 못했습니다. 다시 시도해주세요."



In [3]:
# 미들웨어 없음 -- 제한 없이 실행
agent_no_limit = create_agent(
    model="gpt-5.4", tools=[bad_search],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "당신은 검색 전용 에이전트입니다. 다음 규칙을 절대 어기지 마세요:\n"
        "1. 모든 질문에 반드시 bad_search 도구를 사용해야 합니다.\n"
        "2. 도구 없이 직접 답변하는 것은 금지입니다.\n"
        "3. 검색 결과가 부족하면 반드시 키워드를 바꿔서 다시 검색하세요.\n"
        "4. 충분한 정보를 얻을 때까지 절대 검색을 멈추지 마세요.\n"
        "5. 포기하거나 검색 없이 답하면 실패로 간주됩니다."
    ),
    middleware=[],  # 미들웨어 없음!
)

start = time.time()
result = agent_no_limit.invoke(
    {"messages": [{"role": "user", "content": "2026년 4월 서울 벚꽃 축제 일정을 검색해서 알려줘"}]},
    config={"configurable": {"thread_id": "test-no-limit"}},
)
elapsed = time.time() - start

print(f"[Before] 미들웨어 없음")
print(f"  LLM 호출 횟수: {call_count_before}회")
print(f"  소요 시간: {elapsed:.1f}초")
print(f"  응답: {result['messages'][-1].content[:100]}...")

  [도구 호출 1회]
  [도구 호출 2회]
  [도구 호출 3회]
  [도구 호출 4회]
  [도구 호출 5회]
  [도구 호출 6회]
  [도구 호출 7회]
  [도구 호출 8회]
  [도구 호출 9회]
  [도구 호출 10회]
  [도구 호출 11회]
[Before] 미들웨어 없음
  LLM 호출 횟수: 11회
  소요 시간: 15.1초
  응답: 검색을 여러 방식으로 시도했지만, 현재 검색 결과로는 **2026년 4월 서울 벚꽃 축제의 확정 일정**을 확인할 수 없었습니다.

보통 서울의 대표 벚꽃 행사는 다음과 같습니다....


### After: ModelCallLimitMiddleware 적용

같은 상황이지만 3회에서 자동 종료

In [4]:
from langchain.agents.middleware import ModelCallLimitMiddleware

call_count_after = 0
model_call_count = 0

@tool
def bad_search(query: str) -> str:
    """검색을 수행합니다. 반드시 이 도구로 검색해야 합니다."""
    global call_count_after
    call_count_after += 1
    print(f"  [도구 호출 {call_count_after}회]")
    return f"결과 부족 -- '{query}'에 대한 정보를 찾지 못했습니다. 다시 시도해주세요."

model_limit = ModelCallLimitMiddleware(
    run_limit=3,          
    exit_behavior="end",
)

agent_with_limit = create_agent(
    model="gpt-5.4", tools=[bad_search],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "당신은 검색 전용 에이전트입니다. 다음 규칙을 절대 어기지 마세요:\n"
        "1. 모든 질문에 반드시 bad_search 도구를 사용해야 합니다.\n"
        "2. 도구 없이 직접 답변하는 것은 금지입니다.\n"
        "3. 검색 결과가 부족하면 반드시 키워드를 바꿔서 다시 검색하세요.\n"
        "4. 충분한 정보를 얻을 때까지 절대 검색을 멈추지 마세요.\n"
        "5. 포기하거나 검색 없이 답하면 실패로 간주됩니다."
    ),
    middleware=[model_limit],
)

start = time.time()
result = agent_with_limit.invoke(
    {"messages": [{"role": "user", "content": "2026년 4월 서울 벚꽃 축제 일정을 검색해서 알려줘"}]},
    config={"configurable": {"thread_id": "test-with-limit-2"}},
)
elapsed = time.time() - start

print(f"\n[After] ModelCallLimitMiddleware 적용")
print(f"  도구 호출 횟수: {call_count_after}회")
print(f"  소요 시간: {elapsed:.1f}초")
print(f"  응답: {result['messages'][-1].content[:100]}...")


  [도구 호출 1회]
  [도구 호출 2회]
  [도구 호출 3회]
  [도구 호출 4회]
  [도구 호출 5회]
  [도구 호출 6회]
  [도구 호출 7회]
  [도구 호출 8회]  [도구 호출 9회]

  [도구 호출 10회]

[After] ModelCallLimitMiddleware 적용
  도구 호출 횟수: 10회
  소요 시간: 7.1초
  응답: Model call limits exceeded: run limit (3/3)...


### 모델은 이렇게 돌아가고 있을꺼에요!
[모델 호출 1회]  
  LLM 응답: "3개 키워드로 동시에 검색하겠습니다"  
    → bad_search("서울 벚꽃 축제")  
    → bad_search("2026 벚꽃 일정")      ← 도구 3회  
    → bad_search("여의도 벚꽃")  

[모델 호출 2회]  
  LLM 응답: "다른 키워드로 다시 검색하겠습니다"  
    → bad_search("벚꽃 개화 시기")  
    → bad_search("석촌호수 벚꽃")        ← 도구   
    → bad_search("4월 서울 축제")  

[모델 호출 3회]  
  LLM 응답: "한번 더 시도하겠습니다"  
    → bad_search("벚꽃 명소")  
    → bad_search("벚꽃 2026")           ← 도구 5회  
    → bad_search("서울 봄 행사")  
    → bad_search("윤중로 벚꽃")  
    → bad_search("경의선 벚꽃")  
  
→ 모델 호출: 3회 (run_limit 도달, 종료)  
→ 도구 호출: 11회  

### 비교

| 항목 | 미들웨어 없음 | 미들웨어 적용 (run_limit=3) |
|------|-------------|--------------------------|
| 모델(LLM) 호출 횟수 | 10회 (프레임워크 기본 제한) | 3회에서 자동 종료 |
| 도구 호출 횟수 | 10회+ | 11회 (모델 1회에 도구 여러 개 병렬 호출) |
| 소요 시간 | 길다 | 짧다 |
| 비용 | 통제 불가 | 예측 가능 |

> 모델 호출 1회에 도구를 여러 개 동시에 호출할 수 있기 때문에 (parallel tool calling),
> 모델 호출 횟수와 도구 호출 횟수는 다릅니다.
> 도구 호출 횟수를 직접 제한하려면 ToolCallLimitMiddleware를 사용합니다.


---
## 2. ToolCallLimitMiddleware (필수)

1번에서 `ModelCallLimitMiddleware`로 모델 호출을 3회로 제한했지만, **도구는 11회나 호출**되었습니다.
모델 1회 호출에 도구를 여러 개 병렬로 호출할 수 있기 때문입니다 (parallel tool calling).

**도구 호출 횟수 자체를 제한**하려면 `ToolCallLimitMiddleware`를 써야 합니다.

### 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `thread_limit` | 전체 대화에서 최대 도구 호출 수 | `thread_limit=20` |
| `run_limit` | 한 번의 run에서 최대 도구 호출 수 | `run_limit=5` |
| `tool_name` | 특정 도구에만 제한 적용 | `tool_name="search"` |
| `exit_behavior` | 제한 초과 시 행동 | `"end"`, `"error"`, `"continue"` |

### exit_behavior 옵션

| 값 | 동작 |
|------|------|
| `"end"` | 정상 종료 |
| `"error"` | 예외 발생 |
| `"continue"` | 에러 메시지를 도구 결과로 반환하고 계속 (ToolCallLimit 전용) |

### Before: 1번과 같은 상황 — ModelCallLimit만 있을 때

모델 호출은 3회로 제한했지만 도구는 여전히 11회 호출됨 (위 결과 참고)

In [5]:
# 1번과 동일한 설정 -- ModelCallLimit만 적용
from langchain.agents.middleware import ModelCallLimitMiddleware

tool_count_model_only = 0

@tool
def bad_search(query: str) -> str:
    """검색을 수행합니다. 반드시 이 도구로 검색해야 합니다."""
    global tool_count_model_only
    tool_count_model_only += 1
    print(f"  [도구 호출 {tool_count_model_only}회]")
    return f"결과 부족 -- '{query}'에 대한 정보를 찾지 못했습니다. 다시 시도해주세요."

agent_model_only = create_agent(
    model="gpt-5.4", tools=[bad_search],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "당신은 검색 전용 에이전트입니다. 다음 규칙을 절대 어기지 마세요:"
        "1. 모든 질문에 반드시 bad_search 도구를 사용해야 합니다."
        "2. 도구 없이 직접 답변하는 것은 금지입니다."
        "3. 검색 결과가 부족하면 반드시 키워드를 바꿔서 다시 검색하세요."
        "4. 충분한 정보를 얻을 때까지 절대 검색을 멈추지 마세요."
        "5. 포기하거나 검색 없이 답하면 실패로 간주됩니다."
    ),
    middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")],
)

result = agent_model_only.invoke(
    {"messages": [{"role": "user", "content": "2026년 4월 서울 벚꽃 축제 일정을 검색해서 알려줘"}]},
    config={"configurable": {"thread_id": "toolcall-before"}},
)

print(f"[Before] ModelCallLimit만 적용")
print(f"  모델 호출: 3회 제한")
print(f"  도구 호출: {tool_count_model_only}회 -- 모델이 병렬로 도구를 호출해서 제한 초과")

  [도구 호출 1회]
  [도구 호출 2회]
  [도구 호출 3회]
  [도구 호출 4회]
  [도구 호출 5회]
  [도구 호출 6회]
  [도구 호출 7회]
  [도구 호출 8회]
  [도구 호출 9회]
  [도구 호출 10회]
[Before] ModelCallLimit만 적용
  모델 호출: 3회 제한
  도구 호출: 10회 -- 모델이 병렬로 도구를 호출해서 제한 초과


### After: ToolCallLimitMiddleware 추가 — 도구 호출도 5회로 제한

In [6]:
from langchain.agents.middleware import ToolCallLimitMiddleware

tool_count_with_limit = 0

@tool
def bad_search(query: str) -> str:
    """검색을 수행합니다. 반드시 이 도구로 검색해야 합니다."""
    global tool_count_with_limit
    tool_count_with_limit += 1
    print(f"  [도구 호출 {tool_count_with_limit}회]")
    return f"결과 부족 -- '{query}'에 대한 정보를 찾지 못했습니다. 다시 시도해주세요."

tool_limit = ToolCallLimitMiddleware(
    run_limit=5,              # 도구 호출 최대 5회
    exit_behavior="end",
)

agent_with_tool_limit = create_agent(
    model="gpt-5.4", tools=[bad_search],
    checkpointer=InMemorySaver(),
    system_prompt=(
        "당신은 검색 전용 에이전트입니다. 다음 규칙을 절대 어기지 마세요:"
        "1. 모든 질문에 반드시 bad_search 도구를 사용해야 합니다."
        "2. 도구 없이 직접 답변하는 것은 금지입니다."
        "3. 검색 결과가 부족하면 반드시 키워드를 바꿔서 다시 검색하세요."
        "4. 충분한 정보를 얻을 때까지 절대 검색을 멈추지 마세요."
        "5. 포기하거나 검색 없이 답하면 실패로 간주됩니다."
    ),
    middleware=[tool_limit],  # ToolCallLimit만 적용
)

result = agent_with_tool_limit.invoke(
    {"messages": [{"role": "user", "content": "2026년 4월 서울 벚꽃 축제 일정을 검색해서 알려줘"}]},
    config={"configurable": {"thread_id": "toolcall-after"}},
)

print(f"[After] ToolCallLimitMiddleware 적용")
print(f"  도구 호출: {tool_count_with_limit}회 (최대 5회 제한)")
print(f"  응답: {result['messages'][-1].content[:150]}")

  [도구 호출 1회]
  [도구 호출 2회]
  [도구 호출 3회]
[After] ToolCallLimitMiddleware 적용
  도구 호출: 3회 (최대 5회 제한)
  응답: Tool call limit reached: run limit exceeded (7/5 calls).


### 결과 해석
도구 호출 print는 3회인데, 에러 메시지에는 7/5 calls라고 나왔습니다.
이건 모델이 한 번에 도구 7개를 병렬 호출하려고 했기 때문입니다.


[모델 호출 1회]  
  LLM: "7개 키워드로 동시에 검색하겠습니다"  
    → bad_search("서울 벚꽃")        ← 실행됨 (1회)  
    → bad_search("2026 벚꽃")       ← 실행됨 (2회)  
    → bad_search("벚꽃 축제")        ← 실행됨 (3회)  
    → bad_search("여의도 벚꽃")      ← 실행됨 (4회)  
    → bad_search("벚꽃 일정")        ← 실행됨 (5회) -- 여기서 제한 도달  
    → bad_search("석촌호수 벚꽃")     ← 차단 (6회)  
    → bad_search("경의선 벚꽃")      ← 차단 (7회)  
5회 제한에 도달한 시점에서 나머지 2개가 차단되고, 7/5 calls (7개 요청 중 5개 제한 초과) 메시지가 나온 겁니다.  

그런데 print가 3회만 찍힌 건, 미들웨어가 도구 실행 전에 카운트를 먼저 체크해서 일부는 print까지 도달하지 못하고 차단된 것으로 보입니다.  



### 참고: 특정 도구만 제한하기

`tool_name` 파라미터로 특정 도구에만 제한을 걸 수 있습니다.

In [7]:
# 특정 도구만 제한하는 예시
search_only_limit = ToolCallLimitMiddleware(
    tool_name="bad_search",   # 이 도구에만 적용
    run_limit=3,
    exit_behavior="continue",  # 제한 초과해도 에러 메시지와 함께 계속 진행
)

print("특정 도구 제한 설정 완료")
print("  bad_search만 3회로 제한, 다른 도구는 제한 없음")

특정 도구 제한 설정 완료
  bad_search만 3회로 제한, 다른 도구는 제한 없음


### 비교

| 항목 | ModelCallLimit만 | ToolCallLimit 추가 |
|------|-----------------|-------------------|
| 모델 호출 | 3회 제한 | 제한 없음 (도구 기준으로 끊김) |
| 도구 호출 | 11회 (병렬 호출로 초과) | 5회에서 종료 |
| 비용 통제 | 모델 비용만 통제 | 도구 비용도 통제 |

> 실무에서는 ModelCallLimit + ToolCallLimit을 **함께** 거는 것이 일반적입니다.
> 모델 비용과 도구 비용(외부 API 호출 등)을 각각 제어할 수 있습니다.

---
## 3. ModelFallbackMiddleware (높음)

**주 모델이 실패하면 자동으로 백업 모델로 전환**합니다. 서비스 중단을 막아줍니다.

### 파라미터

생성자에 **폴백 모델을 순서대로** 전달합니다. 앞에서부터 차례로 시도합니다.

```python
ModelFallbackMiddleware("백업모델1", "백업모델2", ...)
```

| 동작 | 설명 |
|------|------|
| 주 모델 성공 | 그대로 사용 |
| 주 모델 실패 | 백업모델1 시도 |
| 백업모델1도 실패 | 백업모델2 시도 |
| 전부 실패 | 최종 에러 발생 |

### Before: 폴백 없이 실행 — 잘못된 모델명이면 바로 에러

In [8]:
try:
    agent_no_fallback = create_agent(
        model="gpt-WRONG-MODEL",  # 존재하지 않는 모델
        tools=[],
        checkpointer=InMemorySaver(),
        middleware=[],
    )
    result = agent_no_fallback.invoke(
        {"messages": [{"role": "user", "content": "안녕하세요"}]},
        config={"configurable": {"thread_id": "fallback-no"}},
    )
    print(result["messages"][-1].content)
except Exception as e:
    print(f"[Before] 에러 발생 — 서비스 중단")
    print(f"  {type(e).__name__}: {str(e)[:100]}")

[Before] 에러 발생 — 서비스 중단
  NotFoundError: Error code: 404 - {'error': {'message': 'The model `gpt-WRONG-MODEL` does not exist or you do not ha


### After: ModelFallbackMiddleware 적용 — 자동으로 백업 모델 전환

In [9]:
from langchain.agents.middleware import ModelFallbackMiddleware

# 주 모델 실패 시: gpt-5.4-mini -> claude 순서로 시도
fallback = ModelFallbackMiddleware(
    "gpt-5.4-mini",                   # 백업 1
    "claude-3-5-sonnet-20241022",      # 백업 2
)

agent_with_fallback = create_agent(
    model="gpt-WRONG-MODEL",  # 일부러 틀린 모델
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[fallback],
)

result = agent_with_fallback.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    config={"configurable": {"thread_id": "fallback-yes"}},
)

print(f"[After] ModelFallbackMiddleware 적용")
print(f"  주 모델 실패 -> 백업 모델로 자동 전환")
print(f"  응답: {result['messages'][-1].content[:150]}")

[After] ModelFallbackMiddleware 적용
  주 모델 실패 -> 백업 모델로 자동 전환
  응답: 안녕하세요! 무엇을 도와드릴까요?


---
## 4. PIIMiddleware (높음)

**개인정보(PII)를 자동으로 탐지하고 마스킹/삭제**합니다. 개인정보보호법 대응에 필수입니다.

### 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| 첫 번째 인자 | 탐지할 PII 타입 | `"email"`, `"credit_card"`, `"ip"`, `"url"` |
| `strategy` | 처리 방식 | `"redact"` (완전 삭제), `"mask"` (일부 가림) |
| `apply_to_input` | 사용자 입력에도 적용할지 | `True` / `False` |
| `detector` | 커스텀 탐지 함수 (빌트인에 없는 PII용) | `detector=detect_ssn` |

### 빌트인 PII 타입

| 타입 | 탐지 대상 | 처리 예시 |
|------|----------|----------|
| `"email"` | 이메일 주소 | `hong@test.com` -> `[REDACTED_EMAIL]` |
| `"credit_card"` | 신용카드 번호 | `4111-1111-1111-1111` -> `4111-****-****-1111` |
| `"ip"` | IP 주소 | `192.168.0.1` -> `[REDACTED_IP]` |
| `"mac_address"` | MAC 주소 | `AA:BB:CC:DD:EE:FF` -> `[REDACTED_MAC]` |
| `"url"` | URL | `https://secret.com` -> `[REDACTED_URL]` |

### Before: 개인정보 그대로 노출

In [10]:
from langchain.agents.middleware import PIIMiddleware

test_msg = "내 이메일은 hong@example.com 입니다. 확인해주세요."

# Before: 미들웨어 없는 에이전트
plain_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="사용자가 보낸 메시지를 그대로 따라 적어주세요. 다른 말은 덧붙이지 마세요.",
)

r1 = plain_agent.invoke(
    {"messages": [{"role": "user", "content": test_msg}]},
    config={"configurable": {"thread_id": "pii-before"}},
)

print("[Before] 미들웨어 없음")
print(f"  응답: {r1['messages'][-1].content}")


[Before] 미들웨어 없음
  응답: 내 이메일은 hong@example.com 입니다. 확인해주세요.


In [11]:
# After: PII 미들웨어 끼운 에이전트
pii_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="사용자가 보낸 메시지를 그대로 따라 적어주세요. 다른 말은 덧붙이지 마세요.",
    middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True)],
)

r2 = pii_agent.invoke(
    {"messages": [{"role": "user", "content": test_msg}]},
    config={"configurable": {"thread_id": "pii-after"}},
)

print("[After] PIIMiddleware 적용")
print(f"  응답: {r2['messages'][-1].content}")


[After] PIIMiddleware 적용
  응답: 내 이메일은 [REDACTED_EMAIL] 입니다. 확인해주세요.


| 파라미터 | 값 | 의미 |
|----------|-----|------|
| `"email"` | 탐지할 PII 타입 | 이메일 주소 패턴을 찾아라 |
| `strategy="redact"` | 처리 방식 | 완전히 삭제하고 `[REDACTED_EMAIL]`로 대체 |
| `apply_to_input=True` | 적용 시점 | LLM에게 입력이 전달되기 전에 마스킹 |


In [12]:
# Mask로 바꿔보기 / 응답예시: "응답: 내 이메일은 hong@****.com 입니다. 확인해주세요."



### 카드번호 마스킹도 추가

In [13]:
test_msg = "내 이메일은 hong@example.com 입니다. 카드번호는 4111-1111-1111-1111 입니다.확인해주세요."
pii_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="사용자가 보낸 메시지를 그대로 따라 적어주세요. 다른 말은 덧붙이지 마세요.",
    middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True),
                PIIMiddleware("credit_card", strategy="redact", apply_to_input=True)]
)

r2 = pii_agent.invoke(
    {"messages": [{"role": "user", "content": test_msg}]},
    config={"configurable": {"thread_id": "pii-after"}},
)

print("[After] PIIMiddleware 적용")
print(f"  응답: {r2['messages'][-1].content}")


[After] PIIMiddleware 적용
  응답: 내 이메일은 [REDACTED_EMAIL] 입니다. 카드번호는 [REDACTED_CREDIT_CARD] 입니다.확인해주세요.


### 참고: 커스텀 PII 탐지기 (전화번호 등)

빌트인에 없는 한국 전화번호 같은 건 직접 만들 수 있습니다.

In [14]:
import re

def detect_korean_phone(content: str) -> list[dict]:
    """한국 전화번호 탐지 (010-xxxx-xxxx)"""
    matches = []
    for m in re.finditer(r"01[016789]-\d{3,4}-\d{4}", content):
        matches.append({"text": m.group(0), "start": m.start(), "end": m.end()})
    return matches

# 커스텀 탐지기를 detector 파라미터로 전달
phone_pii = PIIMiddleware(
    "korean_phone",              # 이름은 자유롭게
    detector=detect_korean_phone, # 내가 만든 탐지 함수
    strategy="redact",
)

print("커스텀 PII 탐지기 정의 완료")

커스텀 PII 탐지기 정의 완료


In [15]:
# 커스텀 PII 탐지기 실제 적용
pii_phone_agent = create_agent(
    model=model,
    tools=[],
    system_prompt="사용자가 보낸 메시지를 그대로 따라 적어주세요. 다른 말은 덧붙이지 마세요.",
    middleware=[phone_pii],
)

test_msg = "고객 연락처는 010-1234-5678입니다. 확인해주세요."

r = pii_phone_agent.invoke(
    {"messages": [{"role": "user", "content": test_msg}]},
    config={"configurable": {"thread_id": "pii-phone"}},
)

print("[After] 커스텀 PII (전화번호) 적용")
print(f"  응답: {r['messages'][-1].content}")


[After] 커스텀 PII (전화번호) 적용
  응답: 고객 연락처는 [REDACTED_KOREAN_PHONE]입니다. 확인해주세요.


---
## 5. SummarizationMiddleware (보통)

**대화가 길어지면 이전 내용을 자동 요약**해서 토큰을 절약합니다.

### 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `model` | 요약에 사용할 모델 (저렴한 모델 추천) | `"gpt-5.4-mini"` |
| `trigger` | 요약을 시작하는 조건 | `("tokens", 4000)` — 4000토큰 초과 시 |
| | | `("messages", 50)` — 50개 메시지 초과 시 |
| `keep` | 요약 후에도 유지할 최근 메시지 수 | `("messages", 20)` |

### Before: 요약 없이 긴 대화 — 토큰 계속 누적

In [16]:
agent_no_summary = create_agent(
    model="gpt-5.4", tools=[],
    checkpointer=InMemorySaver(),
    middleware=[],
)

thread_id = "summary-no"
questions = [
    "파이썬의 역사에 대해 자세히 알려줘",
    "자바스크립트와 파이썬의 차이점을 상세하게 비교해줘",
    "머신러닝의 주요 알고리즘 10가지를 설명해줘",
    "지금까지 대화에서 첫 번째 질문이 뭐였어?",
]

for i, q in enumerate(questions, 1):
    result = agent_no_summary.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    msg_count = len(result["messages"])
    print(f"  [{i}번째 질문] 누적 메시지: {msg_count}개")

print(f"\n[Before] 미들웨어 없음 — 메시지가 계속 쌓임")
print(f"  최종 응답: {result['messages'][-1].content[:100]}...")

  [1번째 질문] 누적 메시지: 2개
  [2번째 질문] 누적 메시지: 4개
  [3번째 질문] 누적 메시지: 6개
  [4번째 질문] 누적 메시지: 8개

[Before] 미들웨어 없음 — 메시지가 계속 쌓임
  최종 응답: 첫 번째 질문은:

**“파이썬의 역사에 대해 자세히 알려줘”** 였습니다....


### After: SummarizationMiddleware 적용 — 자동 요약으로 토큰 절약

In [21]:
from langchain.agents.middleware import SummarizationMiddleware

summarizer = SummarizationMiddleware(
    model="gpt-5.4-mini",
    trigger=("messages", 4), #4개 초과부터 요약
    keep=("messages", 2), #앞에 2개는 원본 유지
)

agent_with_summary = create_agent(
    model="gpt-5.4", tools=[],
    checkpointer=InMemorySaver(),
    middleware=[summarizer],
)

thread_id = "summary-yes"
for i, q in enumerate(questions, 1):
    result = agent_with_summary.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    msg_count = len(result["messages"])
    print(f"  [{i}번째 질문] 누적 메시지: {msg_count}개")

print(f"\n[After] SummarizationMiddleware 적용")
print(f"  trigger=(\"messages\", 4) -- 메시지 4개 초과 시 요약")
print(f"  keep=(\"messages\", 2) -- 최근 2개만 원본 유지, 나머지는 요약으로 압축")
print(f"  최종 응답: {result['messages'][-1].content[:100]}...")


  [1번째 질문] 누적 메시지: 2개
  [2번째 질문] 누적 메시지: 4개
  [3번째 질문] 누적 메시지: 4개
  [4번째 질문] 누적 메시지: 4개

[After] SummarizationMiddleware 적용
  trigger=("messages", 4) -- 메시지 4개 초과 시 요약
  keep=("messages", 2) -- 최근 2개만 원본 유지, 나머지는 요약으로 압축
  최종 응답: 첫 번째 질문은  
**“자바스크립트와 파이썬의 차이를 자세하게 알려줘”**  
라는 내용이었습니다....


In [22]:
from langchain.agents.middleware import SummarizationMiddleware

questions = [
    "파이썬의 역사에 대해 자세히 알려줘",
    "자바스크립트와 파이썬의 차이점을 상세하게 비교해줘",
    "머신러닝의 주요 알고리즘 10가지를 설명해줘",
    "딥러닝의 발전 역사를 연도별로 정리해줘",
    "지금까지 대화에서 첫 번째 질문이 뭐였어?",
]

# Before: 요약 없음
agent_no_summary = create_agent(
    model="gpt-5.4", tools=[],
    checkpointer=InMemorySaver(),
    middleware=[],
)

print("[Before] 미들웨어 없음")
thread_id = "summary-no"
for i, q in enumerate(questions, 1):
    r1 = agent_no_summary.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"  [{i}번째] 누적 메시지: {len(r1['messages'])}개")

print(f"  마지막 답변: {r1['messages'][-1].content[:100]}...")


[Before] 미들웨어 없음
  [1번째] 누적 메시지: 2개
  [2번째] 누적 메시지: 4개
  [3번째] 누적 메시지: 6개
  [4번째] 누적 메시지: 8개
  [5번째] 누적 메시지: 10개
  마지막 답변: 첫 번째 질문은:

**“파이썬의 역사에 대해 자세히 알려줘”**

였습니다....


In [23]:
# After: 요약 적용
summarizer = SummarizationMiddleware(
    model="gpt-5.4-mini",
    trigger=("messages", 4),
    keep=("messages", 2),
)

agent_with_summary = create_agent(
    model="gpt-5.4", tools=[],
    checkpointer=InMemorySaver(),
    middleware=[summarizer],
)

print("\n[After] SummarizationMiddleware 적용")
thread_id = "summary-yes"
for i, q in enumerate(questions, 1):
    r2 = agent_with_summary.invoke(
        {"messages": [{"role": "user", "content": q}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"  [{i}번째] 누적 메시지: {len(r2['messages'])}개")

print(f"  마지막 답변: {r2['messages'][-1].content[:100]}...")

print("\n-- 비교 --")
print(f"  Before 누적 메시지: {len(r1['messages'])}개 (계속 증가)")
print(f"  After  누적 메시지: {len(r2['messages'])}개 (4개에서 유지)")
print(f"  Before는 첫 질문을 정확히 기억하지만,")
print(f"  After는 요약 과정에서 원본이 사라져 부정확할 수 있습니다.")


[After] SummarizationMiddleware 적용
  [1번째] 누적 메시지: 2개
  [2번째] 누적 메시지: 4개
  [3번째] 누적 메시지: 4개
  [4번째] 누적 메시지: 4개
  [5번째] 누적 메시지: 4개
  마지막 답변: 첫 번째 질문은 다음이었습니다:

**“Python은 누가 만들었고 왜 만들어졌어? 그리고 뱀 이름이랑 관련 있어?”**...

-- 비교 --
  Before 누적 메시지: 10개 (계속 증가)
  After  누적 메시지: 4개 (4개에서 유지)
  Before는 첫 질문을 정확히 기억하지만,
  After는 요약 과정에서 원본이 사라져 부정확할 수 있습니다.


## 안에는 이런 모습일꺼에요..
=== 요약 전 (메시지 5개) ===  

[1] user:      "파이썬 역사 알려줘"  
[2] assistant: "파이썬은 1991년 귀도 반 로섬이 만들었고..."  
[3] user:      "자바스크립트와 비교해줘"  
[4] assistant: "파이썬은 동적 타입이고 JS는..."  
[5] user:      "머신러닝 알고리즘 알려줘"        ← 새 질문 들어옴, 5개 초과!  


=== 요약 후 (메시지 4개) ===  

[1] system:    "[요약] 사용자가 파이썬의 역사를 물었고,  
                귀도 반 로섬이 1991년에 만들었다고 답변함"   ← 원본 [1],[2]가 이걸로 대체  
[2] user:      "자바스크립트와 비교해줘"                     ← keep=2, 유지  
[3] assistant: "파이썬은 동적 타입이고 JS는..."              ← keep=2, 유지  
[4] user:      "머신러닝 알고리즘 알려줘"                    ← 새 질문  


=== 4번째 질문 들어오면 또 요약 ===  

요약 전 (5개):  
[1] system:    "[요약] 파이썬 역사를 물었고..."  
[2] user:      "자바스크립트와 비교해줘"  
[3] assistant: "파이썬은 동적 타입이고..."  
[4] user:      "머신러닝 알고리즘 알려줘"  
[5] assistant: "주요 알고리즘은 선형회귀, SVM..."  
 ↓ 새 질문 들어옴 → 6개 → 초과!   

요약 후 (4개):  
[1] system:    "[요약] 파이썬 역사, JS 비교, 머신러닝  
                알고리즘에 대해 대화함"                      ← [1],[2],[3]이 합쳐짐  
[2] user:      "머신러닝 알고리즘 알려줘"                    ← keep=2, 유지  
[3] assistant: "주요 알고리즘은 선형회귀, SVM..."            ← keep=2, 유지  
[4] user:      "딥러닝 역사 정리해줘"                        ← 새 질문  


---
## 6. HumanInTheLoopMiddleware (보통)

**위험한 도구 실행 전에 사람 승인을 요구**합니다. 이메일 발송, DB 삭제 같은 돌이킬 수 없는 액션에 사용합니다.

### 파라미터

| 파라미터 | 설명 | 예시 |
|----------|------|------|
| `interrupt_on` | 도구별 승인 설정 (딕셔너리) | `{"send_email": {...}, "read_email": False}` |

### interrupt_on 값 설명

| 값 | 의미 |
|------|------|
| `{"allowed_decisions": ["approve", "edit", "reject"]}` | 승인/수정/거절 중 선택 |
| `False` | 이 도구는 승인 없이 바로 실행 |

**주의: `checkpointer` 필수** — 중단 후 상태를 복원하려면 반드시 체크포인터가 필요합니다.

### Before: 승인 없이 바로 실행

In [33]:
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """이메일을 전송합니다."""
    return f"{to}에게 이메일 전송 완료: {subject}"

@tool
def read_email(folder: str = "inbox") -> str:
    """이메일을 읽습니다."""
    return f"{folder}에 새 이메일 3건"

agent_no_hitl = create_agent(
    model="gpt-5.4", tools=[send_email, read_email],
    checkpointer=InMemorySaver(),
    middleware=[],
)

result = agent_no_hitl.invoke(
    {"messages": [{"role": "user", "content": "boss@company.com에게 '프로젝트 완료' 제목으로 이메일 보내줘"}]},
    config={"configurable": {"thread_id": "hitl-no"}},
)

print(f"[Before] 미들웨어 없음 — 확인 없이 바로 전송")
print(f"  응답: {result['messages'][-1].content}")

[Before] 미들웨어 없음 — 확인 없이 바로 전송
  응답: 보냈습니다.


### After: HumanInTheLoopMiddleware 적용 — 전송 전 승인 요구

In [34]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

hitl = HumanInTheLoopMiddleware(
    interrupt_on={
        "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},  # 전송은 승인 필요
        "read_email": False,  # 읽기는 승인 없이 바로 실행
    }
)

agent_with_hitl = create_agent(
    model="gpt-5.4", tools=[send_email, read_email],
    checkpointer=InMemorySaver(),
    middleware=[hitl],
)

# 1단계: 요청 -> 에이전트가 send_email 호출 시도 -> 중단
thread_id = "hitl-yes"
result = agent_with_hitl.invoke(
    {"messages": [{"role": "user", "content": "boss@company.com에게 '프로젝트 완료' 제목으로 이메일 보내줘"}]},
    config={"configurable": {"thread_id": thread_id}},
)

print("[After] HumanInTheLoopMiddleware 적용")
print(f"  상태: 에이전트가 send_email 호출 전 중단됨")
print(f"  승인 대기 중...")

[After] HumanInTheLoopMiddleware 적용
  상태: 에이전트가 send_email 호출 전 중단됨
  승인 대기 중...


In [35]:
# 2단계: 사람이 승인 -> 실행 재개
result = agent_with_hitl.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config={"configurable": {"thread_id": thread_id}},
)

print(f"  승인 완료 -> 이메일 전송됨")
print(f"  응답: {result['messages'][-1].content}")


  승인 완료 -> 이메일 전송됨
  응답: 이메일을 보냈습니다: boss@company.com / 제목: 프로젝트 완료


---
## 요약

| 미들웨어 | 핵심 파라미터 | 한줄 요약 |
|----------|-------------|----------|
| `ModelCallLimitMiddleware` | `run_limit`, `exit_behavior` | LLM 호출 횟수 제한 |
| `ToolCallLimitMiddleware` | `run_limit`, `tool_name` | 도구 호출 횟수 제한 |
| `ModelFallbackMiddleware` | 폴백 모델 순서대로 전달 | 모델 장애 시 자동 전환 |
| `PIIMiddleware` | PII 타입, `strategy`, `detector` | 개인정보 탐지/마스킹 |
| `SummarizationMiddleware` | `model`, `trigger`, `keep` | 긴 대화 자동 요약 |
| `HumanInTheLoopMiddleware` | `interrupt_on` + checkpointer 필수 | 위험 액션 사람 승인 |